# Adım 1 — Docker Ortamının Kurulumu ve Doğrulanması

Bu notebook, projenin altyapı katmanını (`docker-compose.yml`) **doğrular**. PDF Adım 1 gereği aşağıdaki servisler ayağa kaldırılmalıdır:

- **Zookeeper** (Kafka koordinasyonu)
- **Kafka Broker** (Streaming mesaj kuyruğu)
- **Python Producer** (CSV → Kafka)
- **Spark Master + Worker** (Distributed processing)
- **Jupyter Lab** (PySpark + Delta + MLflow client)
- **MLflow Tracking Server** (Model deney yönetimi)

## Kurulum Komutları

```bash
# Tüm servisleri build edip başlat
docker compose up -d --build

# Servis durumlarını kontrol et
docker compose ps

# Logları takip et
docker compose logs -f kafka
```

> **Not:** Bu notebook host makinesinde çalıştırılır (Docker socket'e erişim gerekir). İçindeki Python hücreleri `subprocess` ile `docker` CLI'yı çağırır.

## BÖLÜM 1 — Docker ve Compose Sürüm Kontrolü

In [ ]:
import subprocess
import json

def run(cmd: str, capture: bool = True) -> str:
    """Shell komutu çalıştır, çıktıyı string olarak döndür."""
    result = subprocess.run(
        cmd, shell=True, capture_output=capture, text=True
    )
    out = (result.stdout or "") + (result.stderr or "")
    return out.strip()

print("docker --version:")
print(run("docker --version"))
print("\ndocker compose version:")
print(run("docker compose version"))

## BÖLÜM 2 — `docker-compose.yml` İçeriğini Görüntüle

In [ ]:
with open("docker-compose.yml", "r", encoding="utf-8") as f:
    print(f.read())

## BÖLÜM 3 — Servisleri Başlat

In [ ]:
# İlk seferde Spark image'inin build edilmesi 5-10 dakika sürebilir.
# Eğer servisler zaten çalışıyorsa bu komut hızlı geçecektir.
print(run("docker compose up -d --build"))

## BÖLÜM 4 — Servis Durumlarını Listele

**Beklenen:** 7 container (`zookeeper`, `kafka`, `producer`, `spark-master`, `spark-worker`, `jupyter`, `mlflow`) durumu `Up` olmalı.

In [ ]:
print(run("docker compose ps"))

In [ ]:
# JSON formatlı detaylı durum
ps_json = run("docker compose ps --format json")
for line in ps_json.splitlines():
    if not line.strip():
        continue
    try:
        svc = json.loads(line)
        print(f"  {svc.get('Service', '?'):15s} | {svc.get('State', '?'):10s} | {svc.get('Status', '?')}")
    except json.JSONDecodeError:
        pass

## BÖLÜM 5 — Healthcheck ve Bağlantı Testleri

In [ ]:
# 5.1 — Zookeeper sağlık kontrolü
print("▶ Zookeeper (ruok):")
print(run("docker exec zookeeper bash -c 'echo ruok | nc -w 2 localhost 2181'"))
print()

# 5.2 — Kafka topic listesi
print("▶ Kafka topic listesi:")
print(run("docker exec kafka kafka-topics --bootstrap-server localhost:9092 --list"))

In [ ]:
# 5.3 — Spark Master web UI erişilebilirliği (port 8080)
print("▶ Spark Master UI:")
print(run("curl -s -o /dev/null -w 'HTTP %{http_code}' http://localhost:8080"))
print()

# 5.4 — MLflow Tracking Server
print("▶ MLflow Tracking Server:")
print(run("curl -s -o /dev/null -w 'HTTP %{http_code}' http://localhost:5000"))
print()

# 5.5 — Jupyter Lab
print("▶ Jupyter Lab:")
print(run("curl -s -o /dev/null -w 'HTTP %{http_code}' http://localhost:8888"))

## BÖLÜM 6 — Servis URL'leri

| Servis | URL |
|--------|-----|
| Spark Master UI | http://localhost:8080 |
| MLflow Tracking | http://localhost:5000 |
| Jupyter Lab | http://localhost:8888 |
| Kafka Broker (host) | localhost:9092 |
| Kafka Broker (internal) | kafka:29092 |

## BÖLÜM 7 — Servisleri Durdurma

```bash
# Sadece durdur
docker compose stop

# Container'ları sil (volume'ları koru)
docker compose down

# Tümünü sil (volume'lar dahil — DİKKAT: Delta Lake verileri silinir)
docker compose down -v
```